# Experiment 1: Profile Reconstruction Fidelity

**Criterion 1**: Profile Reconstruction R² ≥ 0.7

Evaluates how well the meta-encoder's z-space preserves backbone alignment profiles.
Includes L_info / L_geometry ablations (info-only, geometry-only, combined).

## Colab Setup
Clones the repo, installs dependencies, and mounts Google Drive.

In [ ]:
import os, sys

# 1. Clone repository
REPO_DIR = '/content/trainable-circuits'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/jacobposchl/trainable-circuits {REPO_DIR}

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

# 2. Install extra dependencies
!pip install hdbscan umap-learn -q

# 3. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 4. Path constants
DRIVE_DIR  = '/content/drive/MyDrive/ctls'
DATA_DIR   = DRIVE_DIR + '/data/raw'     # contains cifar-10-batches-py/
CKPT_DIR   = DRIVE_DIR + '/experiments'
CONFIG_DIR = REPO_DIR  + '/configs'

print('Repo:     ', REPO_DIR)
print('Data:     ', DATA_DIR)
print('Checkpts: ', CKPT_DIR)

In [ ]:
import torch
import yaml
import numpy as np
import matplotlib.pyplot as plt
from models.backbone import FrozenBackbone
from models.meta_encoder import MetaEncoder, ProfileRegressor
from data.cifar import get_standard_loaders
from evaluation.circuit_analysis import CircuitAnalyzer
from evaluation.metrics import profile_reconstruction_r2

In [ ]:
# Load config and override data directory
config_path     = CONFIG_DIR + '/phase1.yaml'
checkpoint_path = CKPT_DIR  + '/phase1/best.pt'

with open(config_path) as f:
    config = yaml.safe_load(f)

config['data']['data_dir'] = DATA_DIR

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

In [ ]:
# Build models and load weights
backbone = FrozenBackbone(
    arch=config['model']['arch'],
    num_classes=config['model']['num_classes'],
    pretrained=config['model']['pretrained'],
    pool_mode=config['model'].get('pool_mode', 'gap')
).to(device)

meta_encoder = MetaEncoder(
    layer_dims=backbone.layer_dims,
    projection_dim=config['model']['meta_encoder']['projection_dim'],
    n_heads=config['model']['meta_encoder']['n_heads'],
    n_transformer_layers=config['model']['meta_encoder']['n_transformer_layers'],
    dropout=config['model']['meta_encoder']['dropout']
).to(device)

regressor = ProfileRegressor(
    input_dim=config['model']['meta_encoder']['projection_dim'],
    hidden_dim=config['model']['regressor']['hidden_dim']
).to(device)

ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
meta_encoder.load_state_dict(ckpt['meta_encoder_state'])
regressor.load_state_dict(ckpt['regressor_state'])
meta_encoder.eval()
regressor.eval()
print('Models loaded.')

In [ ]:
# Create validation loader, collect representations, compute pair profiles
_, val_loader = get_standard_loaders(
    data_dir=DATA_DIR,
    batch_size=config['data'].get('batch_size', 256),
    num_workers=0,     # 0 avoids multiprocessing issues in Colab
    augment=False,
    download=False,    # data already on Drive
)

analyzer = CircuitAnalyzer(backbone, meta_encoder, val_loader, device)
data     = analyzer.collect_representations(max_samples=2000)
print(f'Collected {data["labels"].shape[0]} samples')

# Pair indices (subsample to 50k for tractability)
N = data['z_list'][0].shape[0]
idx_a, idx_b = torch.triu_indices(N, N, offset=1)
MAX_PAIRS = 50_000
if idx_a.shape[0] > MAX_PAIRS:
    perm  = torch.randperm(idx_a.shape[0])[:MAX_PAIRS]
    idx_a, idx_b = idx_a[perm], idx_b[perm]

# True pair profiles from backbone trajectories  [N_pairs, L]
true_sims    = CircuitAnalyzer.compute_pair_profiles(data['trajectories'], idx_a, idx_b)
true_sims_np = true_sims.numpy()
L            = len(data['z_list'])
print(f'N pairs: {idx_a.shape[0]}, L layers: {L}')

In [ ]:
# Criterion 1: Profile Reconstruction R²
predicted_sims = []
with torch.no_grad():
    for l in range(L):
        z_prod = data['z_list'][l][idx_a] * data['z_list'][l][idx_b]
        predicted_sims.append(regressor(z_prod.to(device)).cpu())

predicted_np = torch.stack(predicted_sims, dim=1).numpy()

result = profile_reconstruction_r2(predicted_np, true_sims_np)
print(f"R\u00b2 = {result['r2']:.4f}  (target \u2265 0.7, {'PASS' if result['passes'] else 'FAIL'})")

In [ ]:
# Scatter plot: predicted vs true (sample 5k points)
sample = np.random.choice(true_sims_np.size, min(5000, true_sims_np.size), replace=False)
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(true_sims_np.ravel()[sample], predicted_np.ravel()[sample],
           alpha=0.2, s=2, color='steelblue')
ax.plot([-1, 1], [-1, 1], 'r--', lw=1)
ax.set_xlabel('True similarity')
ax.set_ylabel('Predicted similarity')
ax.set_title(f"Profile Reconstruction  R\u00b2={result['r2']:.3f}")
plt.tight_layout()
plt.show()

In [ ]:
# Ablation comparison: info-only vs geometry-only vs combined
# TODO: Update checkpoint paths once ablation experiments are trained
ablation_configs = {
    'info_only':     CONFIG_DIR + '/ablations/info_only.yaml',
    'geometry_only': CONFIG_DIR + '/ablations/geometry_only.yaml',
}
ablation_ckpts = {
    'info_only':     CKPT_DIR + '/ablations/info_only/best.pt',
    'geometry_only': CKPT_DIR + '/ablations/geometry_only/best.pt',
}
# To evaluate: load each model, re-run the prediction cell above, compare R²